# 深度学习专题 01 · 让计算机认出数字和手势

![两组来自课堂训练集的照片，同一个数字和同一个手势各有不同写法或姿态](../assets/cnn_examples.png)

遮住图注，看每一排的三张图。哪些笔画或手指轮廓让你相信它们属于同一类？如果轮廓向右移一格，你还认得出来吗？计算机每次收到的只有 784 个亮度值。

**本次任务**是为两种输入各训练一个全连接网络（Fully Connected Network，本课简称 FCN）和一个卷积神经网络（CNN）。先写下猜想：在哪组图片上，两种模型的测试准确率差距会更大？你看到了什么线索？

按顺序运行。先认图和数像素，再亲手算卷积、搭网络，最后用四行实测结果和一张错例图回答开头的问题。环境中需要 PyTorch。准备单元会显示使用的设备，CUDA 可用时自动选择 NVIDIA 显卡。[安装说明](../README.md#安装与启动)两个真实数据集的课堂子集随项目提供，CPU 即可完成实验。

## 1 · 从一张照片到一个训练样本

一张灰度图可以写成 28 行、28 列的表，每格从 0（黑）到 255（白）。输入网络前把每格除以 255，让数值落在 0 到 1 之间。标签告诉模型这张图应该归到哪一类。

MNIST 的标签是 0–9。Sign Language MNIST 收录美国手语的 24 个静态指拼字母，J 和 Z 需要运动。原标签中 I 是 8，K 是 10。网络的 24 个输出按顺序编号 0–23，所以下面按字母顺序重新编号，显示预测时再查回字母。[MNIST 数据说明](https://keras.io/api/datasets/mnist/) · [手势数据说明](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)

每类用 200 张训练、40 张验证、40 张测试。训练集用来更新权重，验证集用来观察学习进度，最后才打开原测试集抽取的考题。两个模型在各自的数据集上看到完全相同的图片。[抽样与授权说明](data/image_recognition/README.md)

In [ ]:
# 【准备】从项目根目录或本笔记本的目录运行
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import ipywidgets as widgets
from IPython.display import Markdown, display

try:
    from MachineLearning import image_labs as lab
except ModuleNotFoundError as exc:
    if exc.name != "MachineLearning":
        raise
    import image_labs as lab

torch.set_num_threads(min(4, torch.get_num_threads()))
device = lab.training_device()
names = {"MNIST": list("0123456789"), "Sign MNIST": list(lab.LETTERS)}
datasets = {"MNIST": lab.load_digits(), "Sign MNIST": lab.load_signs()}
for name, splits in datasets.items():
    print(name, "训练、验证、测试张数", [len(y) for x, y in splits.values()])

**读图练习**　先自己辨认左图，再看标题里的标签。右图把方框中的像素放大到能读出数值。改一次 `dataset` 和 `sample`，找出相邻亮度突变的位置。它落在笔画或手指的哪个部位？

In [ ]:
dataset, sample = "MNIST", 0  # 试试 "Sign MNIST"
x, y = datasets[dataset]["train"]
img = x[sample, 0]
fig, (ax, zoom) = plt.subplots(1, 2, figsize=(6, 2.8))
ax.imshow(img, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
ax.add_patch(plt.Rectangle((10.5, 10.5), 4, 4, fill=False, edgecolor="coral", lw=2))
ax.set_title(f"{dataset} / {names[dataset][y[sample]]}")
patch = img[11:15, 11:15]
zoom.imshow(patch, cmap="gray", vmin=0, vmax=1)
for r in range(4):
    for c in range(4):
        v = patch[r, c].item()
        zoom.text(c, r, f"{v:.2f}", ha="center", va="center", color="white" if v < .5 else "black")
zoom.set_title("4 x 4 pixels")
for a in (ax, zoom):
    a.axis("off")
plt.show()

## 2 · 第一种读图方法：给每个位置一份权重

FCN 把各行首尾相接成一个长度为 784 的向量。像素顺序仍在。一个隐藏单元给每个位置分配一个权重，把乘积加起来，再加偏置。比如只看两个像素，亮度为 0.2 和 0.8，权重分别为 −1 和 1，偏置为 0，该单元的分数就是 0.6。ReLU 把负分截成 0，正分原样通过。最后一层把多个单元的响应合成每个类别的分数，最高分就是预测。

| 输入 | 展平 | 全连接 + ReLU | 全连接输出 |
| --- | --- | --- | --- |
| 1 × 28 × 28 | 784 个像素 | 64 个隐藏单元 | 每类一个分数 |

**先猜再算**　如果上面两个像素互换位置，经过 ReLU 后还剩多少？若想在另一个位置寻找同一条边，全连接层会自动沿用这两个权重吗？

In [ ]:
# 【练习】交换两格亮度，观察同一单元的响应
pixels = torch.tensor([0.2, 0.8])
weights = torch.tensor([-1., 1.])
for values in (pixels, pixels.flip(0)):
    score = values @ weights
    print([round(v.item(), 1) for v in values], "加权和", round(score.item(), 2), "ReLU 后", round(score.relu().item(), 2))

def build_fcn(classes):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, classes))

assert build_fcn(10)(torch.zeros(2, 1, 28, 28)).shape == (2, 10)

## 3 · 第二种读图方法：让同一把尺子滑过图片

![两行两列的卷积核在竖边上做逐格乘加，同一组权重重复用于整幅图](../assets/cnn_window.png)

看图中高亮的四格。每一行都用右格减左格，两行再求和。窗口跨过从黑到白的竖边时得到 2，落在纯色区域时得到 0。把窗口每次右移一格，走完一行再下移一格，每个答案按窗口位置写下来，就得到一张**特征图**。这正是卷积层所用的滑窗乘加。整张图共用这四个权重，称为**权重共享**。

**先预测再拖动**　窗口左上角在第 0 列时会读到什么？把竖边右移一格，响应峰值会跟着移动，还是消失？在下面的演示里验证（列从 0 数起）。

In [ ]:
# 【演示】左图框住输入窗口，右图框住它写入的一个响应
kernel = np.array([[-1, 1], [-1, 1]])
def edge_image(edge_col):
    image = np.zeros((4, 5), dtype=int)
    image[:, edge_col:] = 1
    return image

def slide_window(col=1, edge_col=2):
    image = edge_image(edge_col)
    patches = np.lib.stride_tricks.sliding_window_view(image, (2, 2))
    response = (patches * kernel).sum(axis=(-2, -1))
    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.5))
    for ax, matrix, width, title in zip(axes, (image, response), (2, 1), ("Input", "Response")):
        ax.imshow(matrix, cmap="Blues", vmin=0, vmax=matrix.max())
        for r, c in np.ndindex(matrix.shape):
            ax.text(c, r, str(matrix[r,c]), ha="center", va="center",
                    color="white" if matrix[r,c] == matrix.max() else "black")
        ax.add_patch(plt.Rectangle((col-.48, -.48), width-.04, width-.04,
                                    fill=False, edgecolor="coral", lw=3))
        ax.set_title(title)
        ax.axis("off")
    plt.show()

widgets.interact(slide_window,
    col=widgets.IntSlider(value=1, min=0, max=3, description="窗口列"),
    edge_col=widgets.IntSlider(value=2, min=1, max=4, description="竖边列"));

**动手写一次乘加**　保留下面的函数接口，尝试用两个循环重写函数体。用 `-kernel` 再试一次：由亮到暗和由暗到亮的边，哪个方向会得到正分？真实 CNN 在训练中学出卷积核的数值，刚才的探测器是我们指定的。

In [ ]:
def window_score(patch, weights):
    return np.sum(patch * weights)

edge = edge_image(2)
assert window_score(edge[:2, :2], kernel) == 0
assert window_score(edge[:2, 1:3], kernel) == 2
assert window_score(edge[:2, 1:3], -kernel) == -2
print("三个窗口检查通过")

## 4 · 把卷积层搭成分类器

一层可以并排学习多个核，每个核产生一张特征图（一个通道）。**最大池化**在每块 2 × 2 区域里保留最大响应，步长为 2，高宽各减半。例如一块响应为 `[[0, 2], [1, 0]]`，池化后留下 2。峰值在这块内挪一格，最大值可能相同。再做一次卷积，后层就能组合前层在相邻位置发现的线索。

| 站点 | 对一张图做什么 | 默认输出形状（通道 × 高 × 宽） |
| --- | --- | --- |
| 输入 | 一张灰度图 | 1 × 28 × 28 |
| 卷积一 → ReLU → 池化 | 8 个 3 × 3 核，四周补一圈零以保留高宽，然后缩小 | 8 × 14 × 14 |
| 卷积二 → ReLU → 池化 | 合并来自 8 个通道的线索，输出 16 张特征图，再缩小 | 16 × 7 × 7 |
| 展平 → 全连接 | 每类得到一个分数 | 10 或 24 |

**搭建练习**　先盖住下一格的函数体，按表格从输入到输出排列 `Conv2d`、`ReLU`、`MaxPool2d`、`Flatten` 和 `Linear`，再和参考实现对照。先在纸上算第一层有多少可训练参数，每个核含一个偏置。再把 `first_channels` 从 8 改为 4，预测哪几层的形状和参数量会变。运行逐层检查，记下解释，再恢复默认值进行对照实验。

In [ ]:
first_channels = 8
def build_cnn(classes):
    return nn.Sequential(
        nn.Conv2d(1, first_channels, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(first_channels, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(16 * 7 * 7, classes),
    )

probe = torch.zeros(2, 1, 28, 28)  # 两张图并成一个批次
network = build_cnn(24)
for layer in network:
    probe = layer(probe)
    count = sum(p.numel() for p in layer.parameters())
    print(f"{type(layer).__name__:12s} {str(tuple(probe.shape)):22s} {count:6d} 个参数")
assert probe.shape == (2, 24)
assert sum(p.numel() for p in network[0].parameters()) == (3 * 3 + 1) * first_channels

## 5 · 让四个模型接受同一套考试

训练时把图片分成小批。网络先给每张图算各类的分数（logits）。交叉熵会在内部把分数换算成概率，真实类别得到的概率越低，损失越大。举例说，真实类别的概率从 0.5 升到 0.8，单张损失就从约 0.69 降到 0.22。反向传播计算每个参数影响损失的梯度，Adam 用它更新参数。遍历一次训练集叫一个 epoch。

**找出四步**　在代码中指出“预测、算损失、求梯度、更新”各在哪一行。为什么每批开始前要清掉上次的梯度？验证时又该跳过哪几步？

同一数据集内固定图像、划分、像素缩放、每轮洗牌顺序、批大小、学习率和轮数。只更换模型，记录参数量。先约定使用最后一轮的权重。比较不同结构时，参数量和计算量也可能一起改变。

训练图每轮刷新。Train 是本轮各批实际训练图的平均交叉熵，Validation 是轮末固定权重后在验证原图上的平均交叉熵。先观察两线是否一起下降，再结合验证准确率解释差距。

In [ ]:
# 【训练】两个模型共用这一个过程
SEED, EPOCHS, BATCH, LR = 42, 8, 64, 0.002

def train_model(build, classes, splits, title):
    torch.manual_seed(SEED)
    shuffle = torch.Generator().manual_seed(SEED)
    model = build(classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    x, y = splits["train"]
    history = lab.TrainingHistory(title)
    for epoch in range(EPOCHS):
        model.train()
        total_loss = torch.zeros((), device=device)
        for idx in torch.randperm(len(y), generator=shuffle).split(BATCH):
            optimizer.zero_grad()
            scores = model(x[idx].to(device))
            loss = loss_fn(scores, y[idx].to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.detach() * len(idx)
        history.record(model, total_loss.item() / len(y), splits["valid"])
    return model, history

runs = {}
for dataset, splits in datasets.items():
    for architecture, build in (("FCN", build_fcn), ("CNN", build_cnn)):
        runs[dataset, architecture] = train_model(
            build, len(names[dataset]), splits, f"{dataset} / {architecture}")
        print(dataset, architecture, "已完成", EPOCHS, "轮")

**读曲线**　左图每个点是该轮每张训练图的平均损失，右图是该轮结束后的验证准确率。先单独读一种颜色，再看两条线何时拉开距离。如果损失持续下降，验证准确率却停住了，增加训练轮数还有多少证据支持？在打开测试结果前记下你的判断。

In [ ]:
fig, axes = plt.subplots(len(datasets), 2, figsize=(8, 5.4), sharex=True)
for row, dataset in enumerate(datasets):
    for arch in ("FCN", "CNN"):
        _, history = runs[dataset, arch]
        for ax, values in zip(axes[row], (history.train_loss, history.valid_accuracy)):
            ax.plot(range(1, len(values)+1), values, ".-", label=arch)
    for ax, metric in zip(axes[row], ("Training loss", "Validation accuracy")):
        ax.set(title=f"{dataset} / {metric}", xlabel="Epoch")
        ax.legend()
    axes[row, 1].set_ylim(0, 1.02)
fig.tight_layout()
plt.show()

### 学出来的卷积核看到了什么？

选一张验证图，让训练好的 CNN 只执行第一层卷积和 ReLU。原图右边是前四个通道的响应，共用同一色标，越亮代表正响应越强。先猜哪个位置会亮，再看指尖、手掌边缘或背景在不同通道中的亮度。换到同一个字母的另一张图，刚才的观察还成立吗？

In [ ]:
# 【观察】修改字母和样本序号，比较同类两张照片
letter, sample_in_class = "V", 0  # 字母可从 names["Sign MNIST"] 中选择，序号 0–39
vx, vy = datasets["Sign MNIST"]["valid"]
index = torch.where(vy == names["Sign MNIST"].index(letter))[0][sample_in_class]
photo = vx[index:index+1]
model, _ = runs["Sign MNIST", "CNN"]
with torch.no_grad():
    maps = model[:2](photo.to(device))[0, :4].cpu()
fig, axes = plt.subplots(1, 1 + len(maps), figsize=(9, 2.1), squeeze=False)
axes = axes[0]
axes[0].imshow(photo[0, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"Input: {letter}")
for i, (ax, response) in enumerate(zip(axes[1:], maps)):
    ax.imshow(response, cmap="magma", vmin=0, vmax=max(maps.max().item(), 1e-6))
    ax.set_title(f"Channel {i+1}")
for ax in axes:
    ax.axis("off")
plt.show()

## 6 · 比较成绩，再回到照片

测试准确率 = 猜对的张数 ÷ 测试总张数。每个数据集内计算 CNN 减 FCN 的**百分点差**。例如从 70% 到 75% 是增加 5 个百分点。先比较同一任务的两行，再回到开头的猜想：哪个数据集的差值更大？

下格用每轮保存的权重在测试原图上回算交叉熵。两幅图的纵轴刻度相同，可以逐轮对照训练与测试损失。测试曲线用于最后复盘；选择轮数和超参数请依据验证集。

In [ ]:
predictions = {}
table = ["| 数据集 | 模型 | 参数量 | 正确 / 测试张数 | 测试准确率 |",
         "| --- | --- | ---: | ---: | ---: |"]
differences = []
for dataset, splits in datasets.items():
    x, y = splits["test"]
    accuracies = {}
    for arch in ("FCN", "CNN"):
        model, _ = runs[dataset, arch]
        model.eval()
        with torch.no_grad():
            pred = model(x.to(device)).argmax(1).cpu()
        predictions[dataset, arch] = pred
        hits = (pred == y).sum().item()
        acc = accuracies[arch] = hits / len(y)
        params = sum(p.numel() for p in model.parameters())
        table.append(f"| {dataset} | {arch} | {params:,} | {hits} / {len(y)} | {acc:.2%} |")
    gap = 100 * (accuracies["CNN"] - accuracies["FCN"])
    differences.append(f"{dataset}：{gap:+.2f} 个百分点")
display(Markdown("\n".join(table)))
print("CNN 减 FCN")
print("；".join(differences))

for dataset, splits in datasets.items():
    compared = {arch: runs[dataset, arch] for arch in ("FCN", "CNN")}
    lab.plot_final_losses(compared, splits["test"], dataset)


**错在哪里？**　下面显示至少有一个模型猜错的测试图。每张图的第一行是真实类别，第二行给出两个模型的预测。换一个数据集，再换一页。选一张有分歧的图，用手指圈出你判断时用的线索。单张错例能说明什么？还有哪些解释需要更多样本验证？

In [ ]:
def show_errors(dataset, page=0):
    x, y = datasets[dataset]["test"]
    fcn, cnn = (predictions[dataset, arch] for arch in ("FCN", "CNN"))
    errors = torch.where((fcn != y) | (cnn != y))[0]
    if not len(errors):
        print("两种模型在这份测试子集上全部答对。")
        return
    pages = (len(errors) + 5) // 6
    page %= pages
    print(f"共 {len(errors)} 张错例，当前第 {page+1}/{pages} 页")
    fig, axes = plt.subplots(1, 6, figsize=(11, 2.1))
    labels = names[dataset]
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, errors[page*6:page*6+6]):
        ax.imshow(x[i, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"True: {labels[y[i]]}\nFCN: {labels[fcn[i]]} | CNN: {labels[cnn[i]]}", fontsize=9)
    plt.show()

widgets.interact(show_errors, dataset=list(datasets), page=widgets.IntText(value=0, description="页码(从0)"));

### 实验记录 · 用证据改写猜想

在下面填写三句话，和同桌交换检查：

1. 我原来猜______。这次在数字上 CNN 与 FCN 相差______个百分点，在手势上相差______个百分点。
2. 在______数据集第______页，我看到真实标签______，两种模型分别猜成______和______。我能从图上指出的线索是______。
3. 我会为______任务选______模型，因为测试准确率______，参数量______。要判断换一批人或换个背景后是否仍然适用，我还需要______。

**讨论边界**　两个任务的类别数和训练张数不同。手势数据还包含同一原图的增强变体，按图片划分的验证集可能比面对新拍摄者更容易。结论要写清是哪些样本、哪套结构、多少轮训练。想进一步检验“共享核有助于应对位移”，可以设计一个只在验证图上做小幅平移的对照，先约定边缘填充值并保持标签不变。

## 选学 · 用 NumPy 复现一整张特征图

把第 3 节的单窗口函数推广到所有合法位置。高 4、宽 5 的输入，用 2 × 2 核、步长 1、不补零时，最后一个窗口从哪一行哪一列开始？先写下输出尺寸，再用下面的双循环验证。最后只把竖边从第 2 列移到第 3 列，预测每行最大值出现在哪里。

In [ ]:
def conv_valid(image, weights):
    h, w = weights.shape
    out = np.empty((image.shape[0]-h+1, image.shape[1]-w+1))
    for r, c in np.ndindex(out.shape):
        out[r, c] = window_score(image[r:r+h, c:c+w], weights)
    return out

for edge_col in (2, 3):
    image = edge_image(edge_col)
    feature = conv_valid(image, kernel)
    reference = nn.functional.conv2d(torch.tensor(image).float()[None, None],
                                     torch.tensor(kernel).float()[None, None])
    np.testing.assert_allclose(feature, reference[0, 0].numpy())
    assert feature.shape == (3, 4)
    print("竖边列", edge_col, "响应峰值列", feature.argmax(axis=1), "\n", feature)